# Código Normalizar Datasets

# Datasets Faixa Etária

In [8]:
import pandas as pd
import re
from functools import reduce
from distritos import agregar_por_distrito

INPUT_DIR  = "../data/processed/"
OUTPUT_DIR = "../data/clean/"

TERRITORY_COL = "Territórios"
year_pattern  = re.compile(r"^(\d{4})")

# Mapeamento ficheiro → nome da coluna de valor (faixa etária)
FICHEIROS = {
    "Dataset 0-19.csv":   "0-19",
    "Dataset 20-34.csv":  "20-34",
    "Dataset 35-64.csv":  "35-64",
    "Dataset 65+.csv":    "65+",
    "Dataset Total.csv":  "Total",
}

In [9]:
dfs = []

for ficheiro, faixa in FICHEIROS.items():
    print(f"\n[A processar] {ficheiro}")
    df = pd.read_csv(INPUT_DIR + ficheiro)

    # Agrupa colunas pelo ano e soma as sub-colunas (ex: "2001 (#3)" → "2001")
    grupos_por_ano: dict[str, list[str]] = {}
    for col in df.columns:
        if col == TERRITORY_COL:
            continue
        m = year_pattern.match(col)
        if m:
            grupos_por_ano.setdefault(m.group(1), []).append(col)

    resultado = pd.DataFrame({TERRITORY_COL: df[TERRITORY_COL]})
    for ano in sorted(grupos_por_ano):
        resultado[ano] = df[grupos_por_ano[ano]].sum(axis=1)

    # Substitui municípios por distritos e soma os valores
    resultado = agregar_por_distrito(resultado, coluna_territorio=TERRITORY_COL)

    # Despivota: anos passam a linhas; coluna de valor recebe o nome da faixa etária
    resultado = resultado.melt(
        id_vars=TERRITORY_COL,
        var_name="Ano",
        value_name=faixa
    )
    resultado["Ano"] = resultado["Ano"].astype(int)
    resultado = resultado.sort_values([TERRITORY_COL, "Ano"]).reset_index(drop=True)

    dfs.append(resultado)
    print(f"Processado: {ficheiro}")

# Junta todos os datasets num só, fazendo merge em Territórios + Ano
dataset_final = reduce(
    lambda left, right: pd.merge(left, right, on=[TERRITORY_COL, "Ano"], how="outer"),
    dfs
)
dataset_final = dataset_final.sort_values([TERRITORY_COL, "Ano"]).reset_index(drop=True)

output_path = OUTPUT_DIR + "Dataset_Combinado.csv"
dataset_final.to_csv(output_path, index=False)
print(f"\nDataset final guardado: {output_path}")
print(f"Shape: {dataset_final.shape}")
print(dataset_final.head())

print("\nConcluído!")


[A processar] Dataset 0-19.csv
Processado: Dataset 0-19.csv

[A processar] Dataset 20-34.csv
Processado: Dataset 20-34.csv

[A processar] Dataset 35-64.csv
Processado: Dataset 35-64.csv

[A processar] Dataset 65+.csv
Processado: Dataset 65+.csv

[A processar] Dataset Total.csv
Processado: Dataset Total.csv

Dataset final guardado: ../data/clean/Dataset_Combinado.csv
Shape: (480, 7)
          Territórios   Ano   0-19  20-34  35-64    65+   Total
0  Distrito da Guarda  2001  35813  32039  66229  45320  179343
1  Distrito da Guarda  2002  34462  32106  66015  45554  178075
2  Distrito da Guarda  2003  33197  31829  65841  45560  176374
3  Distrito da Guarda  2004  31975  31400  65578  45591  174486
4  Distrito da Guarda  2005  30928  30823  65380  45554  172628

Concluído!


# Dataset Mortalidade

In [11]:
df = pd.read_csv(INPUT_DIR + "Dataset Mortalidade.csv")

# Normaliza os nomes das colunas de ano
rename_map = {}
for col in df.columns:
    if col == TERRITORY_COL:
        continue
    m = year_pattern.search(col)
    if m:
        rename_map[col] = m.group(1)
df = df.rename(columns=rename_map)

# Converte taxa por 1000 hab para percentagem
year_cols = [c for c in df.columns if c != TERRITORY_COL]
df[year_cols] = df[year_cols] / 10

# Agrega por distrito usando a média dos municípios
df = agregar_por_distrito(df, coluna_territorio=TERRITORY_COL, agregacao="mean")

# Arredonda após a média por distrito
num_cols = [c for c in df.columns if c != TERRITORY_COL]
df[num_cols] = df[num_cols].round(2)

# Despivota: anos passam a linhas
df = df.melt(
    id_vars=TERRITORY_COL,
    var_name="Ano",
    value_name="Mortalidade (%)"
)
df["Ano"] = df["Ano"].str.extract(r"(\d{4})").astype(int)
df = df.sort_values([TERRITORY_COL, "Ano"]).reset_index(drop=True)

output_path = OUTPUT_DIR + "Dataset_Mortalidade.csv"
df.to_csv(output_path, index=False)
print(f"Guardado: {output_path}")
print(f"Shape: {df.shape}")
print(df.head(10))

print("\nConcluído!")

Guardado: ../data/clean/Dataset_Mortalidade.csv
Shape: (340, 3)
          Territórios   Ano  Mortalidade (%)
0  Distrito da Guarda  2001             1.54
1  Distrito da Guarda  2009             1.66
2  Distrito da Guarda  2010             1.77
3  Distrito da Guarda  2011             1.64
4  Distrito da Guarda  2012             1.76
5  Distrito da Guarda  2013             1.71
6  Distrito da Guarda  2014             1.68
7  Distrito da Guarda  2015             1.86
8  Distrito da Guarda  2016             1.88
9  Distrito da Guarda  2017             1.88

Concluído!
